# Exercise 5.2: Crystal Ball vs Gaussian

From *Programming in High Energy Particle Physics*, Chapter 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch05/ex05_2_solution.ipynb)

Exercise 5.2 Crystal Ball vs Gaussian Implement the Crystal Ball function in ROOT and fit it to a simulated Z boson peak with radiative tails. Compare the fit quality (\(\chi^2/\text{ndf}\)) with a pure Gaussian fit.

<details><summary>Hint</summary>

Write Eq. 5.5 as a C++ function double cb(double* x, double* p) and wrap it in a TF1 , then check it against ROOT's built-in "crystalball" formula. To simulate radiative tails, subtract from each Gaussian-smeared mass a random energy loss, for example drawn from an exponential in a fraction of the events. The Gaussian fit will have \(\chi^2/\text{ndf}\) well above 1 because it cannot follow the low-side tail.

</details>

### Colab setup

The first run takes about 3–5 minutes and restarts the runtime once. After restart, run all cells from the top again. The next cell installs ROOT from conda-forge only if it is missing.

In [ ]:
import sys, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and importlib.util.find_spec("ROOT") is None:
    %pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
import importlib.util
if IN_COLAB and importlib.util.find_spec("ROOT") is None:
    !mamba install -q -y -c conda-forge root
import ROOT
ROOT.gROOT.SetBatch(True)
print("ROOT",ROOT.gROOT.GetVersion())

### Implement the low-side power-law tail and compare fits

In [ ]:
%%writefile ex05_2.C
#include <TF1.h>
#include <TH1D.h>
#include <TCanvas.h>
#include <TRandom3.h>
#include <TStyle.h>
#include <cmath>
#include <algorithm>
#include <iostream>
double cb(double *x,double *p) {
  const double t=(x[0]-p[1])/p[2], a=std::abs(p[3]), n=p[4];
  if(t>-a) return p[0]*std::exp(-0.5*t*t);
  const double A=std::pow(n/a,n)*std::exp(-0.5*a*a);
  const double B=n/a-a;
  return p[0]*A*std::pow(B-t,-n);
}
void ex05_2() {
  TRandom3 rng(42);
  TH1D h("h","Simulated Z peak;Mass [GeV];Events / 0.2 GeV",100,80,100);
  for(int i=0;i<30000;++i) {
    double m=rng.Gaus(91.2,2.0);
    if(rng.Rndm()<0.30) m-=rng.Exp(2.0);
    h.Fill(m);
  }
  TF1 gauss("gauss","gaus",80,100);
  gauss.SetParameters(h.GetMaximum(),91.2,2.0);
  h.Fit(&gauss,"Q0R");
  TF1 custom("custom",cb,80,100,5);
  custom.SetParNames("height","mean","sigma","alpha","n");
  custom.SetParameters(h.GetMaximum(),91.2,2.0,1.5,3.0);
  custom.SetParLimits(2,0.5,5.0); custom.SetParLimits(3,0.2,5.0);
  custom.SetParLimits(4,1.01,20.0);
  h.Fit(&custom,"Q0R");
  TF1 builtin("builtin","crystalball",80,100);
  for(int k=0;k<5;++k) builtin.SetParameter(k,custom.GetParameter(k));
  double maxreldiff=0;
  for(int k=0;k<100;++k) {
    const double x=80.0+0.2*k;
    maxreldiff=std::max(maxreldiff,std::abs(custom.Eval(x)-builtin.Eval(x))/std::max(1.0,custom.Eval(x)));
  }
  std::cout << "Gaussian chi2/ndf=" << gauss.GetChisquare()/gauss.GetNDF() << std::endl;
  std::cout << "Crystal Ball chi2/ndf=" << custom.GetChisquare()/custom.GetNDF() << std::endl;
  std::cout << "max relative difference versus ROOT crystalball=" << maxreldiff << std::endl;
  TCanvas c("c","Z mass",700,500);
  h.SetStats(false);h.Draw("E");
  gauss.SetLineColor(kRed);gauss.Draw("SAME");
  custom.SetLineColor(kBlue);custom.Draw("SAME");
  c.BuildLegend(); c.SaveAs("ex05_2_fit.pdf");
}


In [ ]:
!root -l -b -q ex05_2.C

The radiative-loss toy gives a low-mass tail, so the Crystal Ball typically has a lower χ²/ndf than a Gaussian. It is an approximation: the simulated exponential loss is not exactly a Crystal Ball distribution.